# 04 · An Agent Walks the Golden Thread

Run this **after** notebooks `00` to `03`. In `03` *you* walked the golden thread by hand: a Cypher traversal to find the affected products, a second query to read the schema bridge, then SQL you assembled yourself. Here an **LLM agent does the same job from a single question**.

```
question ──► agent ──► describe_graph_schema / execute_cypher   (Neo4j: who is affected? where does the data live?)
                  └──► execute_sql                              (Snowflake: how much stock do we hold?)
```

The point of the schema bridge (`GraphNodeSchema -[:ENRICHED_BY]-> Table / Column`) is that the agent never has to **guess** a table or column name: the graph tells it where *On-Hand Inventory* lives, and the SQL is built from that answer.

Steps:

1. Load configuration and connect the agent's tools
2. Work out the **expected answer** the manual way (the control)
3. Ask the agent the disruption question
4. **Check the transcript**: did it use the bridge, or guess?

> ✅ This notebook is **read-only**: Neo4j is queried inside read transactions and Snowflake accepts only `SELECT` / `WITH` / `SHOW` / `DESCRIBE`. Nothing is written.
>
> 🤖 LLM output varies between runs, so the exact wording and number of tool calls will differ from run to run. The checks in section 4 tell you whether *your* run took the intended path. On a capable hosted model you should see 3/3 checks pass with a total of 142,442.

## 1 · Configuration and tools

In [ ]:
from pathlib import Path

from workshop_agent.agent import create_text2sql_agent
from workshop_agent.config import load_settings
from workshop_agent.main import _mcp_command  # locates the neocarta-mcp executable
from workshop_agent.tools import make_execute_sql_tool, make_graph_tools
from langchain_mcp_adapters.client import MultiServerMCPClient

env_file = next((p for p in (Path(".env"),) if p.exists()), None)
assert env_file, "No .env found: copy .env.example to .env and fill it in (see docs/attendee_guide.md)"
settings = load_settings(env_file)

assert settings.neo4j["uri"] and settings.snowflake.get("account"), "Neo4j / Snowflake settings missing: run notebook 00 first."
print(f"✅ Config from {env_file}")
print(f"🤖 Agent model (LiteLLM): {settings.agent_model}")

### The agent's toolbox

| Tool | Source | Used for |
|---|---|---|
| `list_schemas`, `get_full_metadata_schema`, metric and business-term search ... | **Neocarta MCP server** | Catalog and semantic layer: what tables, columns and metrics exist |
| `describe_graph_schema`, `execute_cypher` | `workshop_agent.tools` | Read-only access to the supply chain and the schema bridge |
| `execute_sql` | `workshop_agent.tools` | Read-only Snowflake queries |

The Neocarta MCP server is a **catalog** interface. It has no generic Cypher tool, so it cannot walk `Supplier ← Part ← FinishedGood` or read the `ENRICHED_BY` bridge. That is why the workshop adds two small graph tools.

The model is chosen by `AGENT_MODEL` and routed by **LiteLLM**, so you can bring your own: a public API, a local Ollama model or a corporate gateway (see `docs/attendee_guide.md`). For production, prefer maintained servers, such as the official Neo4j Cypher MCP server and a Snowflake-managed MCP server, over these workshop helpers.

In [ ]:
client = MultiServerMCPClient({"sql_metadata_graph": {
    "transport": "stdio", "command": _mcp_command(), "args": [], "env": settings.mcp_env}})
tools = list(await client.get_tools(server_name="sql_metadata_graph"))
tools.append(make_execute_sql_tool(settings.snowflake, settings.row_limit))
tools.extend(make_graph_tools(settings.neo4j, settings.row_limit))
agent = create_text2sql_agent(tools, settings)
print("🔍 Tools:", ", ".join(t.name for t in tools))

## 2 · The expected answer (the control)

Before asking the agent, compute the answer the way notebook `03` did, with our own Cypher and SQL. The agent's answer is checked against this.

"What stock do we hold of the parts they supply?" has two reasonable readings, so we compute both:

* **All stock of the affected parts**: every `PARTSUPP` row for those parts, whichever supplier holds it. This is the stock we can draw on while the supplier is down.
* **The delayed supplier's own stock**: only the rows where `PS_SUPPKEY` is the supplier in trouble.

The question below asks for the total inventory *we currently hold for the affected parts*, which is the first reading.

In [ ]:
import snowflake.connector
from neo4j import GraphDatabase

SUPPLIER_NAME, SUPPLIER_KEY = "Supplier#000000002", "2"

driver = GraphDatabase.driver(settings.neo4j["uri"], auth=settings.neo4j["auth"])
with driver.session(database=settings.neo4j["database"]) as s:
    rows = s.run("""
        MATCH (:Supplier {supplierKey: $k})<-[:SOURCED_FROM]-(p:Part)<-[:REQUIRES*1..2]-(f:FinishedGood)
        RETURN DISTINCT f.sku AS sku, f.name AS name, p.partKey AS part_key ORDER BY sku
    """, k=SUPPLIER_KEY).data()
driver.close()
EXPECTED_SKUS = sorted({r["sku"] for r in rows})
AFFECTED_PARTS = sorted({int(r["part_key"]) for r in rows})
print("Affected finished goods:", {r["sku"]: r["name"] for r in rows})
print("Affected parts:", AFFECTED_PARTS)

conn = snowflake.connector.connect(**settings.snowflake)
cur = conn.cursor()
ids = ",".join(str(k) for k in AFFECTED_PARTS)
cur.execute(f"SELECT SUM(PS_AVAILQTY) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.PARTSUPP WHERE PS_PARTKEY IN ({ids})")
EXPECTED_ALL = int(cur.fetchone()[0])
cur.execute(f"SELECT SUM(PS_AVAILQTY) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.PARTSUPP WHERE PS_PARTKEY IN ({ids}) AND PS_SUPPKEY = {SUPPLIER_KEY}")
EXPECTED_OWN = int(cur.fetchone()[0])
cur.close(); conn.close()
print(f"\n📦 All stock of the affected parts:        {EXPECTED_ALL:,}")
print(f"📦 {SUPPLIER_NAME}'s own stock of them: {EXPECTED_OWN:,}")

## 3 · Ask the agent

The prompt names the supplier, asks for the graph traversal, and tells the agent to use **the schema bridge** to find where inventory lives. It does not name a table or a column.

The cell streams each step: 🌿 a tool call, then 📄 the start of its result.

In [ ]:
PROMPT = (
    "Supplier Supplier#000000002 has just reported a massive production delay. "
    "Use the graph to identify which of our Finished Goods are affected by this delay. "
    "Then, use the graph's schema bridge to find out where 'On-Hand Inventory' is stored for Parts, "
    "and execute a SQL query against Snowflake to tell me exactly how much total inventory we "
    "currently hold for the affected parts."
)

def show(text, limit=500):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[:limit] + " ..."

messages, seen = [], 0
async for chunk in agent.astream(
    {"messages": [{"role": "user", "content": PROMPT}]},
    stream_mode="values",
    config={"configurable": {"thread_id": "module-04"}, "recursion_limit": 60},
):
    messages = chunk["messages"]
    for m in messages[seen:]:
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"🌿 {tc['name']}({show(tc['args'], 400)})")
        elif m.type == "tool":
            print(f"   📄 {show(m.content, 300)}")
    seen = len(messages)

FINAL_ANSWER = str(messages[-1].content)
print("\n" + "=" * 70 + "\n" + FINAL_ANSWER)

## 4 · Did it use the bridge, or guess?

A correct number is not enough: a model that knows TPC-H could guess `PARTSUPP.PS_AVAILQTY` from memory and get the right answer without ever touching the schema bridge. The checks read the **transcript** and verify the path:

| Check | Passes when |
|---|---|
| **C1** | A successful Cypher result lists every affected finished good |
| **C2** | A successful Cypher query on `ENRICHED_BY` returned `PARTSUPP` and `PS_AVAILQTY` **before** the first SQL call |
| **C3** | The SQL ran without error, uses those names, and the final answer contains the expected total |

In [ ]:
import re

calls = {}  # tool_call_id -> {"name", "args"}
steps = []  # ordered (name, args, result) for every tool call
for m in messages:
    for tc in getattr(m, "tool_calls", None) or []:
        calls[tc["id"]] = tc
    if m.type == "tool" and m.tool_call_id in calls:
        tc = calls[m.tool_call_id]
        steps.append((tc["name"], tc["args"], str(m.content)))

ok = lambda result: not result.lstrip().startswith("Error")
cypher_ok = [(a.get("query", ""), r) for n, a, r in steps if n == "execute_cypher" and ok(r)]
sql_steps = [(a.get("query", ""), r) for n, a, r in steps if n == "execute_sql"]
first_sql = next((i for i, (n, _, _) in enumerate(steps) if n == "execute_sql"), len(steps))

checks = []
def check(name, passed, detail=""):
    checks.append(passed)
    print(("✅" if passed else "❌"), name, ("— " + detail) if detail else "")

# C1: the affected finished goods came out of a successful graph query
c1 = any(all(sku in r for sku in EXPECTED_SKUS) for _, r in cypher_ok)
check("C1 Affected finished goods found by graph traversal", c1, ", ".join(EXPECTED_SKUS))

# C2: the bridge (ENRICHED_BY) was queried successfully, and before any SQL ran
bridge = [i for i, (n, a, r) in enumerate(steps)
          if n == "execute_cypher" and ok(r) and "ENRICHED_BY" in a.get("query", "")
          and "partsupp" in r.lower() and "ps_availqty" in r.lower()]
check("C2 Bridge query returned PARTSUPP / PS_AVAILQTY before any SQL", bool(bridge) and bridge[0] < first_sql)

# C3: valid SQL built from the bridge, and the answer carries the expected total
good_sql = [q for q, r in sql_steps if ok(r) and "partsupp" in q.lower() and "ps_availqty" in q.lower()]
numbers = {int(x.replace(",", "")) for x in re.findall(r"\d[\d,]*", FINAL_ANSWER) if x.replace(",", "").isdigit()}
check("C3 SQL executed and answer matches the expected total", bool(good_sql) and EXPECTED_ALL in numbers,
      f"expected {EXPECTED_ALL:,}; SQL calls: {len(sql_steps)}")

print(f"\n{sum(checks)}/{len(checks)} checks passed")

### Reading the result

* **All three pass:** the agent discovered the data location from the graph, then queried it. This is the golden thread done autonomously.
* **C2 fails, C3 passes:** the answer is right, but the agent guessed the table. Look at the 🌿 lines above: weaker models often write Cypher with unnamed relationships such as `-->(x)` and then reference `e.target`. The error text from `execute_cypher` nudges the model to fix this; a stronger model usually recovers on its own.
* **Different total:** check whether the agent summed `PARTSUPP` for every supplier (`EXPECTED_ALL`) or only the delayed supplier's rows (`EXPECTED_OWN`). Both are valid readings; the question decides.

## Where next

The agent here is stateless between runs. A natural extension is an **agent memory layer** in the same graph, so past disruptions and decisions become more context for the next question. This module deliberately writes nothing, to keep the focus on the bridge.